# 03 - Advanced: Processors, DAG Analysis & Flow Optimization

This notebook covers advanced py2dataiku features:

1. **ProcessorCatalog** - Exploring the 100+ available Dataiku processors
2. **Processor Demos** - Building PrepareSteps for key processor types
3. **PrepareSettings** - Composing multi-step Prepare recipes
4. **FlowGraph (DAG)** - Topological sort, cycle detection, subgraph discovery
5. **FlowOptimizer** - Merging recipes, removing orphans, optimization results
6. **Flow Validation** - Structural validation with errors, warnings, info
7. **Column Lineage** - Tracing columns through the flow

In [ ]:
import sys
sys.path.insert(0, "../..")

## 1. ProcessorCatalog

The `ProcessorCatalog` provides metadata for all available Dataiku Prepare recipe processors,
including parameter requirements and example configurations.

In [ ]:
from py2dataiku.mappings.processor_catalog import ProcessorCatalog

# List all available processors
all_processors = ProcessorCatalog.list_processors()
print(f"Total processors available: {len(all_processors)}")
print(f"\nFirst 15 processors: {all_processors[:15]}")

In [ ]:
# List all processor categories
categories = ProcessorCatalog.list_categories()
print("Processor categories:")
for cat in categories:
    processors_in_cat = ProcessorCatalog.list_processors(category=cat)
    print(f"  {cat}: {len(processors_in_cat)} processors")

In [ ]:
# Get detailed info about a specific processor
info = ProcessorCatalog.get_processor("StringTransformer")
print(f"Name: {info.name}")
print(f"Category: {info.category}")
print(f"Description: {info.description}")
print(f"Required params: {info.required_params}")
print(f"Optional params: {info.optional_params}")
print(f"Example params: {info.example_params}")

In [ ]:
# Convenience methods for parameter lookup
print("Required params for Binner:", ProcessorCatalog.get_required_params("Binner"))
print("Example for Formula:", ProcessorCatalog.get_example("Formula"))
print("Example for IfThenElse:", ProcessorCatalog.get_example("IfThenElse"))

## 2. Processor Demos with PrepareStep

Each processor type maps to a `PrepareStep`. Below we demonstrate 15+ key processors
with both the pandas equivalent and the corresponding `PrepareStep` construction.

In [ ]:
from py2dataiku.models.prepare_step import (
    PrepareStep,
    ProcessorType,
    StringTransformerMode,
    NumericalTransformerMode,
    FilterMatchMode,
)

# --- 1. COLUMN_RENAMER ---
# pandas: df.rename(columns={"old_col": "new_col"})
step_rename = PrepareStep.rename_columns({"old_col": "new_col", "amt": "amount"})
print("COLUMN_RENAMER:", step_rename)
print("  Description:", step_rename.get_description())
print()

In [ ]:
# --- 2. FILL_EMPTY_WITH_VALUE ---
# pandas: df["score"].fillna(0)
step_fill = PrepareStep.fill_empty(column="score", value=0)
print("FILL_EMPTY_WITH_VALUE:", step_fill)
print("  Description:", step_fill.get_description())
print()

# --- 3. REMOVE_ROWS_ON_EMPTY ---
# pandas: df.dropna(subset=["email", "name"])
step_dropna = PrepareStep.remove_rows_on_empty(columns=["email", "name"])
print("REMOVE_ROWS_ON_EMPTY:", step_dropna)
print("  Description:", step_dropna.get_description())

In [ ]:
# --- 4. STRING_TRANSFORMER ---
# pandas: df["name"].str.upper()
step_upper = PrepareStep.string_transform(
    column="name",
    mode=StringTransformerMode.UPPERCASE,
)
print("STRING_TRANSFORMER (UPPERCASE):", step_upper)
print("  Description:", step_upper.get_description())
print()

# pandas: df["name"].str.strip()
step_trim = PrepareStep.string_transform(
    column="name",
    mode=StringTransformerMode.TRIM,
)
print("STRING_TRANSFORMER (TRIM):", step_trim)
print()

# Show all available string transformer modes
print("Available StringTransformerMode values:")
for mode in list(StringTransformerMode)[:10]:
    print(f"  {mode.name} -> {mode.value}")

In [ ]:
# --- 5. TYPE_SETTER ---
# pandas: df["id"].astype(int)
step_type = PrepareStep.set_type(column="id", target_type="bigint")
print("TYPE_SETTER:", step_type)
print("  Description:", step_type.get_description())
print()

# --- 6. DATE_PARSER ---
# pandas: pd.to_datetime(df["date_str"], format="%Y-%m-%d")
step_date = PrepareStep.parse_date(column="date_str", formats=["yyyy-MM-dd"])
print("DATE_PARSER:", step_date)

In [ ]:
# --- 7. BINNER ---
# pandas: pd.cut(df["age"], bins=[0, 18, 35, 50, 100])
step_binner = PrepareStep(
    processor_type=ProcessorType.BINNER,
    params={
        "column": "age",
        "mode": "FIXED_BINS",
        "bins": [0, 18, 35, 50, 100],
    },
)
print("BINNER:", step_binner)
print("  Dict:", step_binner.to_dict())
print()

# --- 8. FILTER_ON_VALUE ---
# pandas: df[df["status"] == "active"]
step_filter = PrepareStep.filter_on_value(
    column="status",
    values=["active"],
    matching_mode="EQUALS",
    keep=True,
)
print("FILTER_ON_VALUE:", step_filter)

In [ ]:
# --- 9. ONE_HOT_ENCODER ---
# pandas: pd.get_dummies(df, columns=["category"])
step_ohe = PrepareStep(
    processor_type=ProcessorType.ONE_HOT_ENCODER,
    params={"column": "category"},
)
print("ONE_HOT_ENCODER:", step_ohe)
print()

# --- 10. FORMULA ---
# pandas: df["total"] = df["price"] * df["quantity"]
step_formula = PrepareStep(
    processor_type=ProcessorType.FORMULA,
    params={
        "outputColumn": "total",
        "expression": "price * quantity",
    },
)
print("FORMULA:", step_formula)

In [ ]:
# --- 11. IF_THEN_ELSE ---
# pandas: df["age_group"] = np.where(df["age"] >= 18, "adult", "minor")
step_ite = PrepareStep.if_then_else(
    column="age",
    condition="val >= 18",
    then_value="adult",
    else_value="minor",
    output_column="age_group",
)
print("IF_THEN_ELSE:", step_ite)
print("  Dict:", step_ite.to_dict())
print()

# --- 12. SWITCH_CASE ---
# pandas: df["status"].map({"A": "Active", "I": "Inactive"})
step_switch = PrepareStep.switch_case(
    column="status",
    cases={"A": "Active", "I": "Inactive", "P": "Pending"},
    default_value="Unknown",
    output_column="status_label",
)
print("SWITCH_CASE:", step_switch)

In [ ]:
# --- 13. COLUMN_DELETER ---
# pandas: df.drop(columns=["temp_col", "debug_col"])
step_delete = PrepareStep.delete_columns(["temp_col", "debug_col"])
print("COLUMN_DELETER:", step_delete)
print("  Description:", step_delete.get_description())
print()

# --- 14. REMOVE_DUPLICATES ---
# pandas: df.drop_duplicates(subset=["id"])
step_dedup = PrepareStep.remove_duplicates(columns=["id"])
print("REMOVE_DUPLICATES:", step_dedup)
print("  Description:", step_dedup.get_description())

In [ ]:
# --- 15. REGEXP_EXTRACTOR ---
# pandas: df["zip"] = df["address"].str.extract(r"(\d{5})")
step_regex = PrepareStep.regexp_extract(
    column="address",
    pattern=r"(\d{5})",
    output_columns=["zip_code"],
)
print("REGEXP_EXTRACTOR:", step_regex)
print()

# --- 16. NUMERICAL_TRANSFORMER ---
# pandas: df["amount"] = df["amount"] * 100
step_num = PrepareStep(
    processor_type=ProcessorType.NUMERICAL_TRANSFORMER,
    params={"column": "amount", "mode": "MULTIPLY", "value": 100},
)
print("NUMERICAL_TRANSFORMER:", step_num)
print()

# --- 17. TRANSLATE_VALUES ---
# pandas: df["country"].replace({"US": "United States", "UK": "United Kingdom"})
step_translate = PrepareStep.translate_values(
    column="country_code",
    translations={"US": "United States", "UK": "United Kingdom", "FR": "France"},
)
print("TRANSLATE_VALUES:", step_translate)

In [ ]:
# PrepareStep serialization round-trip
original = PrepareStep.if_then_else(
    column="score", condition="val > 90",
    then_value="A", else_value="B", output_column="grade",
)
as_dict = original.to_dict()
restored = PrepareStep.from_dict(as_dict)
print("Original:", original)
print("Dict:    ", as_dict)
print("Restored:", restored)
print("Match:   ", original.processor_type == restored.processor_type
      and original.params == restored.params)

## 3. PrepareSettings - Multi-Step Prepare Recipes

`PrepareSettings` composes multiple `PrepareStep` objects into a single recipe configuration.
This is how Dataiku's Prepare recipe works: an ordered list of transformation steps.

In [ ]:
from py2dataiku.models.recipe_settings import PrepareSettings
from py2dataiku.models.dataiku_recipe import DataikuRecipe, RecipeType

# Build a multi-step cleaning pipeline as a single Prepare recipe
cleaning_steps = [
    PrepareStep.remove_rows_on_empty(columns=["email"]),
    PrepareStep.string_transform("name", StringTransformerMode.TRIM),
    PrepareStep.string_transform("email", StringTransformerMode.LOWERCASE),
    PrepareStep.fill_empty("age", 0),
    PrepareStep.set_type("age", "bigint"),
    PrepareStep.rename_columns({"nm": "full_name"}),
    PrepareStep.delete_columns(["temp", "debug"]),
]

settings = PrepareSettings(steps=cleaning_steps)
print(f"PrepareSettings with {len(settings.steps)} steps")
print(f"Display dict: {settings.to_display_dict()}")

In [ ]:
# Create a DataikuRecipe with composed settings
recipe = DataikuRecipe(
    name="clean_customers",
    recipe_type=RecipeType.PREPARE,
    inputs=["raw_customers"],
    outputs=["cleaned_customers"],
    steps=cleaning_steps,
    settings=settings,
)
print(f"Recipe: {recipe.name}")
print(f"Type: {recipe.recipe_type.value}")
print(f"Steps: {len(recipe.steps)}")
for i, step in enumerate(recipe.steps):
    print(f"  {i+1}. {step.get_description()}")

## 4. FlowGraph - DAG Analysis

The `FlowGraph` class provides a directed acyclic graph (DAG) representation of a Dataiku flow.
It supports topological sorting, cycle detection, path finding, and connected component discovery.

In [ ]:
from py2dataiku.models.flow_graph import FlowGraph, NodeType

# Build a graph manually to demonstrate the API
graph = FlowGraph()

# Add dataset nodes
graph.add_node("raw_data", NodeType.DATASET)
graph.add_node("cleaned_data", NodeType.DATASET)
graph.add_node("features", NodeType.DATASET)
graph.add_node("ref_table", NodeType.DATASET)
graph.add_node("joined_data", NodeType.DATASET)
graph.add_node("final_output", NodeType.DATASET)

# Add recipe nodes
graph.add_node("recipe:clean", NodeType.RECIPE)
graph.add_node("recipe:featurize", NodeType.RECIPE)
graph.add_node("recipe:join", NodeType.RECIPE)
graph.add_node("recipe:aggregate", NodeType.RECIPE)

# Add edges: dataset -> recipe -> dataset
graph.add_edge("raw_data", "recipe:clean")
graph.add_edge("recipe:clean", "cleaned_data")
graph.add_edge("cleaned_data", "recipe:featurize")
graph.add_edge("recipe:featurize", "features")
graph.add_edge("features", "recipe:join")
graph.add_edge("ref_table", "recipe:join")
graph.add_edge("recipe:join", "joined_data")
graph.add_edge("joined_data", "recipe:aggregate")
graph.add_edge("recipe:aggregate", "final_output")

print(graph)

In [ ]:
# Explore node types
print("Dataset nodes:")
for node in graph.dataset_nodes:
    print(f"  {node.name}")

print("\nRecipe nodes:")
for node in graph.recipe_nodes:
    print(f"  {node.name}")

print(f"\nTotal nodes: {len(graph)}")
print(f"Total edges: {len(graph.edges)}")

In [ ]:
# Topological sort - execution order
topo_order = graph.topological_sort()
print("Topological order (execution sequence):")
for i, name in enumerate(topo_order):
    node = graph.get_node(name)
    print(f"  {i+1}. [{node.node_type.value}] {name}")

In [ ]:
# Successors and predecessors
print("Successors of 'cleaned_data':", graph.get_successors("cleaned_data"))
print("Predecessors of 'cleaned_data':", graph.get_predecessors("cleaned_data"))
print()
print("Successors of 'recipe:join':", graph.get_successors("recipe:join"))
print("Predecessors of 'recipe:join':", graph.get_predecessors("recipe:join"))

In [ ]:
# Roots and leaves
print("Root nodes (sources):", graph.get_roots())
print("Leaf nodes (sinks):", graph.get_leaves())

In [ ]:
# Path finding
path = graph.get_path("raw_data", "final_output")
print("Path from raw_data to final_output:")
print("  " + " -> ".join(path))

# No path exists between disconnected nodes
no_path = graph.get_path("final_output", "raw_data")
print(f"\nPath from final_output to raw_data: {no_path}")

In [ ]:
# Cycle detection - this graph is acyclic
cycles = graph.detect_cycles()
print(f"Cycles detected: {len(cycles)}")
print(f"Graph is a valid DAG: {len(cycles) == 0}")

In [ ]:
# Connected components (subgraphs)
subgraphs = graph.find_disconnected_subgraphs()
print(f"Number of connected components: {len(subgraphs)}")
for i, component in enumerate(subgraphs):
    print(f"  Component {i+1}: {sorted(component)}")

In [ ]:
# Demonstrate disconnected subgraphs by adding an isolated pipeline
graph2 = FlowGraph()

# Pipeline A
graph2.add_node("ds_a1", NodeType.DATASET)
graph2.add_node("recipe:r_a", NodeType.RECIPE)
graph2.add_node("ds_a2", NodeType.DATASET)
graph2.add_edge("ds_a1", "recipe:r_a")
graph2.add_edge("recipe:r_a", "ds_a2")

# Pipeline B (disconnected from A)
graph2.add_node("ds_b1", NodeType.DATASET)
graph2.add_node("recipe:r_b", NodeType.RECIPE)
graph2.add_node("ds_b2", NodeType.DATASET)
graph2.add_edge("ds_b1", "recipe:r_b")
graph2.add_edge("recipe:r_b", "ds_b2")

subgraphs2 = graph2.find_disconnected_subgraphs()
print(f"Disconnected graph has {len(subgraphs2)} components:")
for i, comp in enumerate(subgraphs2):
    print(f"  Component {i+1}: {sorted(comp)}")

### FlowGraph from a DataikuFlow

The `DataikuFlow.graph` property automatically builds a `FlowGraph` from the flow's recipes and datasets.

In [ ]:
from py2dataiku import convert

# Convert a multi-step pipeline to a flow
code = """
import pandas as pd

df = pd.read_csv('sales.csv')
df = df.dropna()
df = df.rename(columns={'amt': 'amount'})
df['amount'] = df['amount'].astype(float)
result = df.groupby('region').agg({'amount': 'sum'})
result.to_csv('regional_sales.csv')
"""

flow = convert(code)

# Access the DAG through the graph property
dag = flow.graph
print(dag)
print(f"\nDataset nodes: {len(dag.dataset_nodes)}")
print(f"Recipe nodes: {len(dag.recipe_nodes)}")
print(f"\nTopological order:")
for name in dag.topological_sort():
    print(f"  {name}")

### Visualizing the Flow DAG

Let's visualize the flow we just built to see how the DAG looks in different formats.
We use ASCII for quick terminal reference, Mermaid for documentation, and SVG for
pixel-accurate Dataiku-style rendering.

In [ ]:
# ASCII visualization of the flow DAG
print(flow.visualize(format="ascii"))

In [ ]:
# Mermaid visualization (GitHub/Notion compatible)
print(flow.visualize(format="mermaid"))

In [ ]:
# SVG visualization renders inline in Jupyter
from IPython.display import SVG, display

svg_content = flow.visualize(format="svg")
display(SVG(svg_content))

In [ ]:
# PlantUML visualization for technical specs
print(flow.visualize(format='plantuml'))

## 5. FlowOptimizer

The `FlowOptimizer` can merge consecutive Prepare recipes, remove orphan intermediate datasets,
and generate recommendations for improving flow performance.

In [ ]:
from py2dataiku.models.dataiku_flow import DataikuFlow
from py2dataiku.models.dataiku_recipe import DataikuRecipe, RecipeType
from py2dataiku.models.dataiku_dataset import DataikuDataset, DatasetType
from py2dataiku.optimizer.flow_optimizer import FlowOptimizer, OptimizationResult

# Build a flow with two consecutive Prepare recipes (suboptimal)
flow_before = DataikuFlow(name="unoptimized_flow")

# Datasets
flow_before.datasets = [
    DataikuDataset(name="raw_input", dataset_type=DatasetType.INPUT),
    DataikuDataset(name="intermediate_1", dataset_type=DatasetType.INTERMEDIATE),
    DataikuDataset(name="intermediate_2", dataset_type=DatasetType.INTERMEDIATE),
    DataikuDataset(name="final_output", dataset_type=DatasetType.OUTPUT),
]

# Recipe 1: first Prepare step
recipe1 = DataikuRecipe(
    name="prepare_step1",
    recipe_type=RecipeType.PREPARE,
    inputs=["raw_input"],
    outputs=["intermediate_1"],
    steps=[
        PrepareStep.remove_rows_on_empty(columns=["email"]),
        PrepareStep.string_transform("name", StringTransformerMode.TRIM),
    ],
)

# Recipe 2: second Prepare step (consecutive, mergeable)
recipe2 = DataikuRecipe(
    name="prepare_step2",
    recipe_type=RecipeType.PREPARE,
    inputs=["intermediate_1"],
    outputs=["intermediate_2"],
    steps=[
        PrepareStep.fill_empty("age", 0),
        PrepareStep.set_type("age", "bigint"),
    ],
)

# Recipe 3: Grouping (not mergeable with Prepare)
recipe3 = DataikuRecipe(
    name="compute_group_stats",
    recipe_type=RecipeType.GROUPING,
    inputs=["intermediate_2"],
    outputs=["final_output"],
    group_keys=["region"],
)

flow_before.recipes = [recipe1, recipe2, recipe3]

print("BEFORE OPTIMIZATION:")
print(f"  Recipes: {len(flow_before.recipes)}")
print(f"  Datasets: {len(flow_before.datasets)}")
for r in flow_before.recipes:
    step_info = f" ({len(r.steps)} steps)" if r.steps else ""
    print(f"  - {r.name} [{r.recipe_type.value}]{step_info}: {r.inputs} -> {r.outputs}")

In [ ]:
# Run the optimizer
optimizer = FlowOptimizer()
optimized_flow = optimizer.optimize(flow_before, apply=True)
result = optimizer.last_result

print("AFTER OPTIMIZATION:")
print(f"  Recipes: {len(optimized_flow.recipes)}")
print(f"  Datasets: {len(optimized_flow.datasets)}")
for r in optimized_flow.recipes:
    step_info = f" ({len(r.steps)} steps)" if r.steps else ""
    print(f"  - {r.name} [{r.recipe_type.value}]{step_info}: {r.inputs} -> {r.outputs}")

print(f"\nOptimizationResult:")
print(f"  Recipes merged: {result.recipes_merged}")
print(f"  Datasets removed: {result.datasets_removed}")
print(f"  Log:")
for entry in result.log:
    print(f"    - {entry}")

### Before vs After: Visual Comparison

Let's see the flow structure before optimization.

In [ ]:
# Visualize the flow BEFORE optimization (rebuild the unoptimized flow)
flow_viz_before = DataikuFlow(name='before_optimization')
flow_viz_before.datasets = [
    DataikuDataset(name='raw_input', dataset_type=DatasetType.INPUT),
    DataikuDataset(name='intermediate_1', dataset_type=DatasetType.INTERMEDIATE),
    DataikuDataset(name='intermediate_2', dataset_type=DatasetType.INTERMEDIATE),
    DataikuDataset(name='final_output', dataset_type=DatasetType.OUTPUT),
]
flow_viz_before.recipes = [
    DataikuRecipe(
        name='prepare_step1', recipe_type=RecipeType.PREPARE,
        inputs=['raw_input'], outputs=['intermediate_1'],
        steps=[PrepareStep.remove_rows_on_empty(columns=['email']),
               PrepareStep.string_transform('name', StringTransformerMode.TRIM)],
    ),
    DataikuRecipe(
        name='prepare_step2', recipe_type=RecipeType.PREPARE,
        inputs=['intermediate_1'], outputs=['intermediate_2'],
        steps=[PrepareStep.fill_empty('age', 0), PrepareStep.set_type('age', 'bigint')],
    ),
    DataikuRecipe(
        name='compute_group_stats', recipe_type=RecipeType.GROUPING,
        inputs=['intermediate_2'], outputs=['final_output'],
        group_keys=['region'],
    ),
]

print('=== BEFORE OPTIMIZATION ===')
print(flow_viz_before.visualize(format='ascii'))

In [ ]:
# Visualize the flow AFTER optimization
print('=== AFTER OPTIMIZATION ===')
print(optimized_flow.visualize(format='ascii'))

# Also show as Mermaid for comparison
print('\nMermaid (after optimization):')
print(optimized_flow.visualize(format='mermaid'))

In [ ]:
# OptimizationResult as a dict
print("Result dict:", result.to_dict())

# Optimization notes stored on the flow
print("\nFlow optimization notes:")
for note in optimized_flow.optimization_notes:
    print(f"  - {note}")

In [ ]:
# Recommendation-only mode (no changes applied)
flow_recommend = DataikuFlow(name="recommend_test")
flow_recommend.datasets = [
    DataikuDataset(name="src", dataset_type=DatasetType.INPUT),
    DataikuDataset(name="mid", dataset_type=DatasetType.INTERMEDIATE),
    DataikuDataset(name="dst", dataset_type=DatasetType.OUTPUT),
]
flow_recommend.recipes = [
    DataikuRecipe(
        name="prep_a", recipe_type=RecipeType.PREPARE,
        inputs=["src"], outputs=["mid"],
        steps=[PrepareStep.fill_empty("x", 0)],
    ),
    DataikuRecipe(
        name="prep_b", recipe_type=RecipeType.PREPARE,
        inputs=["mid"], outputs=["dst"],
        steps=[PrepareStep.set_type("x", "bigint")],
    ),
]

opt2 = FlowOptimizer()
opt2.optimize(flow_recommend, apply=False)  # recommendation only

print(f"Recipes unchanged: {len(flow_recommend.recipes)} (still 2)")
print(f"Recommendations generated: {len(flow_recommend.recommendations)}")
for rec in flow_recommend.recommendations:
    print(f"  [{rec.priority}] {rec.type}: {rec.message}")
    if rec.action:
        print(f"    Action: {rec.action}")

## 6. Flow Validation

The `flow.validate()` method performs structural validation using DAG analysis.
It checks for cycles, orphan datasets, missing references, disconnected subgraphs,
and Python fallback recipes.

In [ ]:
# Validate a well-formed flow
good_flow = DataikuFlow(name="valid_flow")
good_flow.datasets = [
    DataikuDataset(name="input_ds", dataset_type=DatasetType.INPUT),
    DataikuDataset(name="output_ds", dataset_type=DatasetType.OUTPUT),
]
good_flow.recipes = [
    DataikuRecipe(
        name="transform", recipe_type=RecipeType.PREPARE,
        inputs=["input_ds"], outputs=["output_ds"],
        steps=[PrepareStep.fill_empty("x", 0)],
    )
]

result = good_flow.validate()
print("Valid flow:")
print(f"  valid: {result['valid']}")
print(f"  errors: {result['errors']}")
print(f"  warnings: {result['warnings']}")
print(f"  info: {result['info']}")

In [ ]:
# Validate a flow with issues
bad_flow = DataikuFlow(name="problematic_flow")

# Add an orphan dataset (not connected to any recipe)
bad_flow.datasets = [
    DataikuDataset(name="input_ds", dataset_type=DatasetType.INPUT),
    DataikuDataset(name="orphan_ds", dataset_type=DatasetType.INPUT),
    DataikuDataset(name="output_ds", dataset_type=DatasetType.OUTPUT),
    DataikuDataset(name="branch_out", dataset_type=DatasetType.OUTPUT),
]

# Recipes forming two disconnected subgraphs + a Python fallback
bad_flow.recipes = [
    DataikuRecipe(
        name="transform", recipe_type=RecipeType.PREPARE,
        inputs=["input_ds"], outputs=["output_ds"],
        steps=[PrepareStep.fill_empty("x", 0)],
    ),
    DataikuRecipe(
        name="py_fallback", recipe_type=RecipeType.PYTHON,
        inputs=["branch_out"], outputs=[],
        code="custom_transform(df)",
    ),
]

result = bad_flow.validate()
print("Problematic flow:")
print(f"  valid: {result['valid']}")
print(f"  errors ({len(result['errors'])}):")
for e in result['errors']:
    print(f"    [{e['type']}] {e['message']}")
print(f"  warnings ({len(result['warnings'])}):")
for w in result['warnings']:
    print(f"    [{w['type']}] {w['message']}")
print(f"  info ({len(result['info'])}):")
for i in result['info']:
    print(f"    [{i['type']}] {i['message']}")

## 7. Column Lineage

The `flow.get_column_lineage()` method traces a column backward through the flow's recipes
to find its origin dataset and any transformations applied along the way.

In [ ]:
from py2dataiku.models.dataiku_flow import DataikuFlow, ColumnLineage

# Build a flow with trackable column transformations
lineage_flow = DataikuFlow(name="lineage_demo")
lineage_flow.datasets = [
    DataikuDataset(name="source_data", dataset_type=DatasetType.INPUT),
    DataikuDataset(name="cleaned_data", dataset_type=DatasetType.INTERMEDIATE),
    DataikuDataset(name="final_data", dataset_type=DatasetType.OUTPUT),
]

# Prepare recipe with several column transformations
lineage_flow.recipes = [
    DataikuRecipe(
        name="clean_step",
        recipe_type=RecipeType.PREPARE,
        inputs=["source_data"],
        outputs=["cleaned_data"],
        steps=[
            PrepareStep.fill_empty("revenue", 0),
            PrepareStep.string_transform("name", StringTransformerMode.UPPERCASE),
        ],
    ),
    DataikuRecipe(
        name="finalize",
        recipe_type=RecipeType.PREPARE,
        inputs=["cleaned_data"],
        outputs=["final_data"],
        steps=[
            PrepareStep(
                processor_type=ProcessorType.ROUND_COLUMN,
                params={"column": "revenue", "precision": 2},
            ),
        ],
    ),
]

# Trace the "revenue" column
lineage = lineage_flow.get_column_lineage("revenue", dataset="final_data")
print(f"Column: {lineage.column}")
print(f"Final dataset: {lineage.final_dataset}")
print(f"Origin dataset: {lineage.origin_dataset}")
print(f"Origin column: {lineage.origin_column}")
print(f"Transformations ({len(lineage.transformations)}):")
for t in lineage.transformations:
    print(f"  - {t}")

In [ ]:
# Lineage as a dict (for serialization)
print("Lineage dict:")
import json
print(json.dumps(lineage.to_dict(), indent=2))

In [ ]:
# Trace the "name" column through a different transformation path
name_lineage = lineage_flow.get_column_lineage("name", dataset="final_data")
print(f"Column 'name' lineage:")
print(f"  Origin: {name_lineage.origin_dataset}.{name_lineage.origin_column}")
print(f"  Final:  {name_lineage.final_dataset}.{name_lineage.column}")
print(f"  Transformations: {name_lineage.transformations}")

## 8. End-to-End Example: Convert, Analyze, Optimize

Putting it all together: convert Python code, inspect the DAG, validate, and optimize.

In [ ]:
from py2dataiku import convert
from py2dataiku.optimizer.flow_optimizer import FlowOptimizer

# A more complex pipeline
pipeline_code = """
import pandas as pd
import numpy as np

# Load and clean
df = pd.read_csv('transactions.csv')
df = df.dropna(subset=['amount', 'customer_id'])
df = df.rename(columns={'amt': 'amount', 'cust_id': 'customer_id'})
df['amount'] = df['amount'].astype(float)

# Aggregate by customer
summary = df.groupby('customer_id').agg({'amount': 'sum', 'date': 'count'})
summary = summary.rename(columns={'date': 'transaction_count'})

# Top customers
top = summary.nlargest(100, 'amount')
"""

flow = convert(pipeline_code, optimize=False)
print("Flow summary (before optimization):")
print(flow.get_summary())

In [ ]:
# DAG analysis
dag = flow.graph
print(f"DAG: {dag}")
print(f"Roots: {dag.get_roots()}")
print(f"Leaves: {dag.get_leaves()}")
print(f"Cycles: {dag.detect_cycles()}")
print(f"Components: {len(dag.find_disconnected_subgraphs())}")

In [ ]:
# Validate
validation = flow.validate()
print(f"Valid: {validation['valid']}")
if validation['warnings']:
    print("Warnings:")
    for w in validation['warnings']:
        print(f"  - {w}")

In [ ]:
# Optimize
optimizer = FlowOptimizer()
optimizer.optimize(flow, apply=True)
result = optimizer.last_result

print("Optimization result:")
print(f"  Recipes merged: {result.recipes_merged}")
print(f"  Datasets removed: {result.datasets_removed}")
for entry in result.log:
    print(f"  - {entry}")

print(f"\nFlow summary (after optimization):")
print(flow.get_summary())

In [ ]:
# Visualize the optimized end-to-end flow
print('=== Optimized Pipeline (ASCII) ===')
print(flow.visualize(format='ascii'))

print('\n=== Optimized Pipeline (Mermaid) ===')
print(flow.visualize(format='mermaid'))

print('\n=== Optimized Pipeline (PlantUML) ===')
print(flow.visualize(format='plantuml'))

## Summary

This notebook covered py2dataiku's advanced capabilities:

| Feature | Key Classes/Methods | Purpose |
|---------|--------------------|---------|
| **ProcessorCatalog** | `list_processors()`, `get_processor()`, `list_categories()` | Browse 100+ Dataiku processor types with metadata |
| **PrepareStep** | `ProcessorType`, factory methods, `to_dict()`/`from_dict()` | Create individual transformation steps |
| **PrepareSettings** | `PrepareSettings(steps=[...])` | Compose multi-step Prepare recipes |
| **FlowGraph** | `topological_sort()`, `detect_cycles()`, `find_disconnected_subgraphs()` | DAG analysis of flow structure |
| **FlowOptimizer** | `optimize()`, `OptimizationResult` | Merge recipes, remove orphans, generate recommendations |
| **Validation** | `flow.validate()` | Structural checks: cycles, orphans, missing refs |
| **Column Lineage** | `flow.get_column_lineage()` | Trace columns through transformations to their origin |